<a href="https://colab.research.google.com/github/CSB-KaracaLab/af-distogram-flexibility/blob/main/AF_distogram_Colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold distograms and hinge motions, in Colab

**Companion notebook to [`CSB-KaracaLab/af-distogram-flexibility`](https://github.com/CSB-KaracaLab/af-distogram-flexibility).**

AlphaFold's distogram head predicts, for every residue pair, a distribution over distances
rather than one number. That distribution can be broad where the final coordinates look
perfectly confident — and in a protein that hinges on binding, the breadth sits exactly
across the hinge. This notebook walks that idea end to end on one of the paper's own
systems, and leaves you set up to do the same on yours.

> Savaş B, Barlas AB, Karaca E. *Exploring the Potential of AlphaFold Distograms for
> Predicting Binding-induced Hinge Motions.* **FEBS Letters** ·
> [doi:10.1002/1873-3468.70297](https://doi.org/10.1002/1873-3468.70297)

### The worked example: AK2, unbound

**AK2** is the paper's main case. It hinges when AIFM1 binds it, and the hinge shows up in
the Cβ–Cβ distance between residues **230 and 233**. Molecular dynamics puts that distance
at a median of **5.9 Å in the apo protein** and **10.0 Å in the complex** — a 4 Å opening
on binding.

Everything predicted here is of **AK2 alone**. The question the paper asks is whether a
prediction of the unbound protein already knows about the bound conformation, and the
distogram is where to look: the final coordinates collapse to one structure, but the
distogram keeps the whole distribution.

The notebook:

1. downloads the AK2 ensembles the paper deposited on
   [Zenodo](https://doi.org/10.5281/zenodo.17638774) — 4,292 models of the apo protein
   from nine sampling methods;
2. measures the 230–233 distance in every one with the repository's own script, and
   **checks the result against the table the paper deposited**;
3. puts that spread, the AlphaFold distogram, and the apo and complex MD references on one
   axis.

Step 2 is a real reproduction, not a demonstration: if the numbers stop matching, the
notebook says so.

### What you need

A Colab CPU runtime. No GPU — nothing here runs AlphaFold; it reads what AlphaFold
produced. The download is about 143 MB and the measurement takes a few seconds.

In [ ]:
#@title 1. Set up { display-mode: "form" }
#@markdown Clones the repository and installs numpy, pandas and matplotlib. Quick: the
#@markdown distance measurements need nothing beyond the standard library.
import os, subprocess, sys, time

t0 = time.time()
if not os.path.isdir("af-distogram-flexibility"):
    subprocess.run(["git", "clone", "--quiet", "--depth", "1",
                    "https://github.com/CSB-KaracaLab/af-distogram-flexibility.git"],
                   check=True)
REPO = os.path.abspath("af-distogram-flexibility")
SCRIPTS = os.path.join(REPO, "scripts")

if not os.path.isfile("DEPS_READY"):
    os.system(f"pip install -q -r {REPO}/requirements.txt matplotlib")
    os.system("touch DEPS_READY")

import numpy, pandas  # noqa: E402
print(f"repository : {REPO}")
print(f"numpy      : {numpy.__version__}")
print(f"pandas     : {pandas.__version__}")
print(f"ready in {time.time() - t0:.0f} s")

In [ ]:
#@title 2. Fetch the deposited AK2 ensembles { display-mode: "form" }
#@markdown 143 MB from Zenodo: 4,292 predicted structures of apo AK2 from nine sampling
#@markdown methods — AF2 and AF3 enhanced sampling, and the generative models the paper
#@markdown compares against.
import collections, os, urllib.request, zipfile
from pathlib import Path

ZENODO = "https://zenodo.org/records/17638774/files/AK2.zip"
archive = Path("AK2.zip")

if not archive.exists():
    print("downloading ...", flush=True)
    urllib.request.urlretrieve(ZENODO, archive)
print(f"{archive} : {archive.stat().st_size / 1e6:.0f} MB")

if not Path("AK2").is_dir():
    with zipfile.ZipFile(archive) as z:
        z.extractall(".")

ENSEMBLES = Path("AK2")
counts = collections.Counter(p.parent.name for p in ENSEMBLES.rglob("*.pdb"))
print(f"\n{sum(counts.values())} structures in {len(counts)} ensembles:")
for name, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<14} {n:>5}")

In [ ]:
#@title 3. Measure residues 230 and 233 in all of them { display-mode: "form" }
#@markdown This is the repository's own `dist-calc-two-res.py`, pointed at the directory.
#@markdown It reads the coordinates straight out of the PDB files, so the whole ensemble
#@markdown takes a few seconds.
residue1 = 230  #@param {type:"integer"}
residue2 = 233  #@param {type:"integer"}
atom = "CB"  #@param ["CB", "CA"]
#@markdown Glycine has no CB. If either residue is a glycine, measure on CA instead.
import subprocess, time
from pathlib import Path

out_csv = Path(f"AK2_res{residue1}_{residue2}_recomputed.csv")
if out_csv.exists():
    out_csv.unlink()                      # the script appends, so start clean

t0 = time.time()
run = subprocess.run(
    [sys.executable, f"{SCRIPTS}/dist-calc-two-res.py",
     "--input_pdb", str(ENSEMBLES),
     "--residue1", str(residue1), "--residue2", str(residue2),
     "--atom", atom, "--output_csv", str(out_csv)],
    capture_output=True, text=True)
print(run.stdout.strip() or run.stderr[-1500:])
print(f"\n{time.time() - t0:.0f} s")

In [ ]:
#@title 4. Does it reproduce the deposited table? { display-mode: "form" }
#@markdown The paper deposited its own measurements for this pair. Recomputing them from
#@markdown the deposited structures should give the same numbers back; this cell says
#@markdown whether it does, rather than assuming it.
import numpy as np, pandas as pd
from pathlib import Path

mine = pd.read_csv(out_csv)
deposited_path = Path(REPO) / "data/pdb-distances/AK2_res230_233_CB_measurements.csv"

if (residue1, residue2, atom) != (230, 233, "CB") or not deposited_path.is_file():
    print(f"No deposited table for {atom}{residue1}-{atom}{residue2}; "
          "the comparison below only applies to the published CB230-CB233 pair.")
    measured = mine
else:
    deposited = pd.read_csv(deposited_path).rename(columns={"Distance": "deposited"})
    merged = mine.merge(deposited[["PDB file", "deposited"]], on="PDB file", how="outer",
                        indicator=True)
    both = merged[merged._merge == "both"].copy()
    both["difference"] = (both.Distance - both.deposited).abs()

    print(f"recomputed here     : {len(mine)}")
    print(f"deposited with paper: {len(deposited)}")
    print(f"matched by filename : {len(both)}")
    print(f"only here           : {(merged._merge == 'left_only').sum()}")
    print(f"only deposited      : {(merged._merge == 'right_only').sum()}")
    print()
    agree = (both.difference < 0.01).sum()
    print(f"agree within 0.01 A : {agree} / {len(both)}")
    print(f"largest difference  : {both.difference.max():.4f} A")
    if agree == len(both):
        print("\nEvery deposited measurement reproduces from the deposited structures.")
    else:
        worst = both.nlargest(5, "difference")[["PDB file", "Distance", "deposited",
                                                "difference"]]
        print("\nLargest disagreements:")
        display(worst)
    measured = mine

In [ ]:
#@title 5. How far apart do the methods put them? { display-mode: "form" }
#@markdown One dot per predicted structure, by the method that produced it. Every one of
#@markdown these is a prediction of AK2 **alone**. The two dashed lines are where molecular
#@markdown dynamics puts the same distance in the apo protein and in the AIFM1 complex.
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

INK, INK_SOFT, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
HUE = "#2a78d6"            # one hue: the methods are positions on the axis, not a palette
APO, COMPLEX = "#2a78d6", "#eb6834"


def md_reference(state):
    """The MD distances for one state, from data/MD/<pair>_<state>_merged.dist."""
    path = Path(REPO) / f"data/MD/{residue1}_{residue2}_{state}_merged.dist"
    if not path.is_file():
        return None
    out = []
    for n, line in enumerate(path.read_text().splitlines()):
        if n == 0 or not line.strip():
            continue
        parts = line.split(",")
        if len(parts) == 2:
            out.append(float(parts[1]))
    return np.array(out)


MD = {state: md_reference(state) for state in ("apo", "complex")}
for state, values in MD.items():
    if values is not None:
        print(f"MD {state:<8}: {len(values)} frames, median {np.median(values):.2f} A, "
              f"10th-90th {np.percentile(values, 10):.2f}-{np.percentile(values, 90):.2f} A")

order = (measured.groupby("Folder").Distance.median().sort_values().index.tolist())
fig, ax = plt.subplots(figsize=(7.8, 0.46 * len(order) + 1.9), dpi=140)
rng = np.random.default_rng(0)

for state, colour in (("apo", APO), ("complex", COMPLEX)):
    if MD[state] is None:
        continue
    ax.axvline(np.median(MD[state]), color=colour, lw=1.6, ls="--", zorder=2)
    ax.axvspan(np.percentile(MD[state], 10), np.percentile(MD[state], 90),
               color=colour, alpha=0.07, zorder=1)

for y, method in enumerate(order):
    values = measured.loc[measured.Folder == method, "Distance"].to_numpy()
    ax.scatter(values, y + rng.uniform(-0.17, 0.17, len(values)), s=9, alpha=0.35,
               color=INK_SOFT, linewidths=0, zorder=3)
    ax.plot([np.median(values)] * 2, [y - 0.3, y + 0.3], color=INK, lw=2,
            solid_capstyle="round", zorder=4)

label_x = ax.get_xlim()[1]
for y, method in enumerate(order):
    ax.text(label_x, y, f"  n={int((measured.Folder == method).sum())}", va="center",
            fontsize=8, color=INK_SOFT)

top_of_axes = len(order) - 0.4
for state, colour in (("apo", APO), ("complex", COMPLEX)):
    if MD[state] is not None:
        ax.text(np.median(MD[state]), top_of_axes, f" MD {state}", color=colour,
                fontsize=8.5, va="bottom", ha="left")

ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=9)
ax.set_xlabel(f"{atom}{residue1}-{atom}{residue2} distance (A)", fontsize=9.5, color=INK_SOFT)
ax.set_title(f"AK2 predicted alone: {len(measured)} structures, "
             "the black bar is each method's median",
             fontsize=10.5, color=INK, pad=16, loc="left")
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=INK_SOFT, length=0)
plt.tight_layout()
plt.show()

summary = (measured.groupby("Folder").Distance
           .agg(n="count", median="median", spread=lambda s: s.max() - s.min())
           .reindex(order).round(2))
if MD["complex"] is not None:
    threshold = np.percentile(MD["complex"], 10)
    reach = (measured.groupby("Folder").Distance
             .apply(lambda s: round(100 * (s >= threshold).mean(), 1))
             .reindex(order))
    summary[f"% at or past {threshold:.1f} A"] = reach
display(summary)

In [ ]:
#@title 6. Does the distogram know about the bound state? { display-mode: "form" }
#@markdown This is the paper's question in one figure. The predictions are all of apo AK2,
#@markdown and they cluster where MD puts the apo protein. The distogram is the same
#@markdown prediction's own distance distribution - and the thing to look for is whether
#@markdown it carries probability out at the complex distance, which no single predicted
#@markdown structure shows.
import pandas as pd

dgram_path = Path(REPO) / "data/distograms/AK2_distograms_res230_233.csv"
dgram = pd.read_csv(dgram_path)
axis = dgram["Distance_Angstrom"].to_numpy()
runs = dgram.drop(columns=["Distance_Angstrom"])
print(f"{runs.shape[1]} predictions, {runs.shape[0]} distance bins")

def method_of(column):
    return column.split("-")[0].split("_")[0]

families = sorted({method_of(c) for c in runs.columns})
mean_curve = runs.mean(axis=1).to_numpy()

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7.8, 5.6), dpi=140, sharex=True,
                                  gridspec_kw={"height_ratios": [1.15, 1]})

for state, colour in (("apo", APO), ("complex", COMPLEX)):
    if MD[state] is None:
        continue
    for ax_ in (top, bottom):
        ax_.axvline(np.median(MD[state]), color=colour, lw=1.6, ls="--", zorder=2)
        ax_.axvspan(np.percentile(MD[state], 10), np.percentile(MD[state], 90),
                    color=colour, alpha=0.07, zorder=1)

for family in families:
    cols = [c for c in runs.columns if method_of(c) == family]
    top.plot(axis, runs[cols].mean(axis=1), lw=1.1, alpha=0.45, color=INK_SOFT, zorder=3)
top.plot(axis, mean_curve, lw=2.4, color=INK, zorder=4,
         label="distogram, mean over all predictions")
top.set_ylabel("distogram probability", fontsize=9.5, color=INK_SOFT)
# "center right" keeps the legend clear of the MD labels along the top edge
top.legend(frameon=False, fontsize=8.5, labelcolor=INK_SOFT, loc="center right")
top.set_title("AK2 residues 230-233: predicted unbound, against the MD references",
              fontsize=10.5, color=INK, pad=16, loc="left")
for state, colour in (("apo", APO), ("complex", COMPLEX)):
    if MD[state] is not None:
        top.text(np.median(MD[state]), top.get_ylim()[1], f" MD {state}", color=colour,
                 fontsize=8.5, va="top", ha="left")

bins = np.arange(axis.min(), axis.max() + 0.3125, 0.3125)
bottom.hist(measured.Distance, bins=bins, color=INK_SOFT, alpha=0.55, zorder=3,
            label="predicted structures")
bottom.set_ylabel("structures", fontsize=9.5, color=INK_SOFT)
bottom.set_xlabel("C-beta to C-beta distance (A)", fontsize=9.5, color=INK_SOFT)
bottom.legend(frameon=False, fontsize=8.5, labelcolor=INK_SOFT, loc="upper right")

for ax_ in (top, bottom):
    ax_.set_axisbelow(True)
    ax_.yaxis.grid(True, color=GRID, lw=0.8)
    for side in ("top", "right"):
        ax_.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax_.spines[side].set_color(GRID)
    ax_.tick_params(colors=INK_SOFT, length=0)

# the distogram axis runs to 22 A, but nearly all of the signal sits well below that
upper = max(axis[mean_curve.cumsum() / mean_curve.sum() < 0.999].max(),
            measured.Distance.quantile(0.995),
            np.percentile(MD["complex"], 99) if MD["complex"] is not None else 0) + 1.0
top.set_xlim(axis.min(), min(upper, axis.max()))
plt.tight_layout()
plt.show()

peak = axis[int(np.argmax(mean_curve))]
cumulative = np.cumsum(mean_curve) / mean_curve.sum()
lo = axis[int(np.searchsorted(cumulative, 0.10))]
hi = axis[int(np.searchsorted(cumulative, 0.90))]
print(f"distogram : most likely {peak:.2f} A, central 80% spans {lo:.2f}-{hi:.2f} A")
print(f"predicted : median {measured.Distance.median():.2f} A, "
      f"10th-90th {measured.Distance.quantile(0.1):.2f}-"
      f"{measured.Distance.quantile(0.9):.2f} A")

if MD["complex"] is not None:
    bound = np.percentile(MD["complex"], 10)
    in_dgram = float(mean_curve[axis >= bound].sum() / mean_curve.sum())
    in_structures = float((measured.Distance >= bound).mean())
    print(f"\nAt or beyond {bound:.2f} A, where the complex sits:")
    print(f"  distogram probability there : {100 * in_dgram:.1f}%")
    print(f"  predicted structures there  : {100 * in_structures:.1f}%")
    print("\nThe distogram placing more weight on the bound distance than the predicted "
          "coordinates ever visit is the signal this paper is about.")

In [ ]:
#@title 7. A distogram from your own AlphaFold run { display-mode: "form" }
#@markdown Optional. Upload an AlphaFold output holding distogram logits — ColabFold's
#@markdown `*.pickle`, an AlphaFold `result_model_*.pkl`, or an `.npz` — and pull out any
#@markdown residue pair. Leave it unticked to skip.
upload_a_prediction = False  #@param {type:"boolean"}
pair_a = 230  #@param {type:"integer"}
pair_b = 233  #@param {type:"integer"}

if upload_a_prediction:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    run = subprocess.run([sys.executable, f"{SCRIPTS}/extract_distogram_pair.py",
                          "--input_pickle", name, str(pair_a), str(pair_b)],
                         capture_output=True, text=True)
    print(run.stdout.strip() or run.stderr.strip())
    csv_out = Path(f"{Path(name).stem}_res{pair_a}_{pair_b}.csv")
    if csv_out.is_file():
        own = pd.read_csv(csv_out)
        fig, ax = plt.subplots(figsize=(7, 2.9), dpi=140)
        ax.plot(own.Distance_Angstrom, own.Probability, lw=2, color=HUE)
        ax.fill_between(own.Distance_Angstrom, 0, own.Probability, color=HUE, alpha=0.12)
        ax.set_xlabel("distance (Å)", fontsize=9.5, color=INK_SOFT)
        ax.set_ylabel("probability", fontsize=9.5, color=INK_SOFT)
        ax.set_title(f"{name}: residues {pair_a}–{pair_b}", fontsize=10.5, loc="left",
                     color=INK, pad=10)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(GRID)
        ax.tick_params(colors=INK_SOFT, length=0)
        ax.yaxis.grid(True, color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        plt.tight_layout()
        plt.show()
else:
    print("Skipped. Tick the box above to upload one of your own predictions.")

In [ ]:
#@title 8. Perturb an MSA by masking columns { display-mode: "form" }
#@markdown The third script replays a column-masking perturbation onto an alignment. In
#@markdown the paper the columns came from an AFsample2 run, which records them in its
#@markdown output pickle; `--columns` lets you give them directly, which is what this
#@markdown cell does since no pickle is deposited with the repository.
columns_to_mask = "5,6,7,20,21,22"  #@param {type:"string"}

msa_in = Path(REPO) / "data/msas/ak2_0.a3m"
run = subprocess.run([sys.executable, f"{SCRIPTS}/column_masking.py",
                      "--msa_file", str(msa_in),
                      "--columns", columns_to_mask,
                      "--output", "ak2_perturbed.a3m"],
                     capture_output=True, text=True)
print(run.stdout.strip() or run.stderr.strip())

before = msa_in.read_text().splitlines()
after = Path("ak2_perturbed.a3m").read_text().splitlines()
print("\nthe first few rows, before and after:")
shown = 0
for b, a in zip(before, after):
    if b.startswith((">", "#")):
        continue
    print(f"  before  {b[:72]}")
    print(f"  after   {a[:72]}")
    shown += 1
    if shown == 3:
        break
print("\nThe query (the first sequence) is deliberately left alone, and lowercase letters "
      "are insertions that occupy no alignment column, so they are stepped over when the "
      "columns are located in each row.")

## Using this on your own system

Everything above is driven by three arguments: a structure or a directory of them, and a
residue pair. To run it on your own protein:

1. **Measure the pair across your ensemble.** Point step 3 at a directory of PDB files —
   AlphaFold models, frames from a trajectory, any PDB file. One row per model comes
   out, so the result is a distribution. If either residue is a glycine, set `atom`
   to `CA`: glycine has no Cβ.
2. **Pull the distogram out of the prediction.** Step 7 takes a ColabFold `.pickle`, an
   AlphaFold `result_model_*.pkl`, or an `.npz`. For a **complex**, remember that the
   distogram runs over the chains concatenated — a residue number from the second chain is
   not its index; add the lengths of the chains before it.
3. **Compare the two.** A distogram that is broad where the sampled structures are also
   broad is the signal the paper is after.

### What is not here

No AlphaFold run. This notebook reads predictions; it does not make them. The enhanced
sampling the paper compares — MinnieFold, AF_cluster, AFsample2, AF3sample2 — each have
their own repositories, and [MinnieFold](https://github.com/CSB-KaracaLab/MinnieFold) is
the lab's own.

### Citing

> Savaş B, Barlas AB, Karaca E. *Exploring the Potential of AlphaFold Distograms for
> Predicting Binding-induced Hinge Motions.* **FEBS Letters** ·
> [doi:10.1002/1873-3468.70297](https://doi.org/10.1002/1873-3468.70297)

The structure ensembles are deposited separately:
[10.5281/zenodo.17638774](https://doi.org/10.5281/zenodo.17638774).